In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

PLL_DIR = Path.cwd().resolve().parents[2] / "results" / "pll"  # notebook lives in sw/tools/notebooks/
RESULT_CSV = None  # e.g. PLL_DIR / "pll_spectrum_kvco_S5_20261005_140920.csv"; None = newest run

if RESULT_CSV is None:
    RESULT_CSV = sorted((p for p in PLL_DIR.glob("pll_spectrum_kvco_*.csv") if not p.stem.endswith("_traces")),
                        key=lambda p: p.stat().st_mtime)[-1]
RESULT_CSV = Path(RESULT_CSV)
TRACE_CSV = RESULT_CSV.with_name(RESULT_CSV.stem + "_traces.csv")
runs = pd.read_csv(RESULT_CSV).reset_index(drop=True)
for col in ("time_jitter_noise_ps", "time_jitter_spur_ps", "n_spurs", "spurs", "carrier_dbm"):  # older runs
    if col not in runs:
        runs[col] = np.nan
traces = pd.read_csv(TRACE_CSV) if TRACE_CSV.exists() else pd.DataFrame(columns=["f_vco", "kvco_target", "kind"])
curve_f = traces.loc[traces["kind"] == "L_trace", "freq"]
band = (curve_f.min(), curve_f.max())  # integration band of the full curve [Hz]

# Only locked settings: without lock there is no carrier and the measured "phase noise" is the analyzer noise
unlocked = runs[runs["locked"].fillna(0) != 1]
runs = runs[runs["locked"].fillna(0) == 1].reset_index(drop=True)
print(f"{RESULT_CSV.name}: {len(runs) + len(unlocked)} settings, {len(runs)} locked "
      f"({(runs['status'] == 'ok').sum()} measured ok); traces: {TRACE_CSV.name if len(traces) else 'none'}")
if len(unlocked):
    print("excluded (not locked): " + ", ".join(
        f"|Kvco| {r.kvco_pred / 1e6:.0f} MHz/V (c{r.vco_tune_coarse:.0f} min {r.vco_current_min:.0f} "
        f"max {r.vco_current_max:.0f})" for r in unlocked.itertuples()))

pd.DataFrame({
    "f_vco [MHz]": runs["f_vco"] / 1e6,
    "f_pad [MHz]": runs["f_out_meas"] / 1e6,  # analyzer centre frequency = carrier on the pad
    "setting": [f"c{r.vco_tune_coarse:.0f} min {r.vco_current_min:.0f} max {r.vco_current_max:.0f}"
                if not np.isnan(r.vco_tune_coarse) else "-" for r in runs.itertuples()],
    "|Kvco| target [MHz/V]": runs["kvco_target"] / 1e6,
    "|Kvco| pred [MHz/V]": runs["kvco_pred"] / 1e6,
    "Vctrl pred [V]": runs["vctrl_pred"],
    "f_vco meas [MHz]": runs["f_vco_meas"] / 1e6,
    "N": runs["total_div"],
    "I [mA]": runs["pll_i"] * 1e3,
    "P [mW]": runs["pll_p"] * 1e3,
    "carrier [dBm]": runs["carrier_dbm"],
    "jitter curve [ps]": runs["time_jitter_trace_ps"],
    "jitter noise [ps]": runs["time_jitter_noise_ps"],
    "jitter spurs [ps]": runs["time_jitter_spur_ps"],
    "spurs": runs["spurs"],
    "status": runs["status"],
}).round(3)

pll_spectrum_kvco_S5_20261020_113911.csv: 8 settings, 4 locked (4 measured ok); traces: pll_spectrum_kvco_S5_20261020_113911_traces.csv
excluded (not locked): |Kvco| 62 MHz/V (c10 min 0 max 10), |Kvco| 96 MHz/V (c10 min 0 max 0), |Kvco| 9232 MHz/V (c6 min 11 max 0), |Kvco| 21142 MHz/V (c2 min 14 max 0)


,f_vco [MHz],f_pad [MHz],setting,|Kvco| target [MHz/V],|Kvco| pred [MHz/V],Vctrl pred [V],f_vco meas [MHz],N,I [mA],P [mW],carrier [dBm],jitter curve [ps],jitter noise [ps],jitter spurs [ps],spurs,status
0,1000.0,62.5,c5 min 8 max 6,328.942,460.975,0.661,999.999,16,NaN,NaN,1.360,493.937,312.984,382.119,4818 Hz: -75.0 dBc; 88920 Hz: -54.0 dBc; 35820...,ok
1,1000.0,62.5,c5 min 8 max 0,756.354,772.328,0.677,999.999,16,NaN,NaN,1.359,582.831,319.275,487.601,89700 Hz: -53.2 dBc; 360000 Hz: -47.7 dBc; 555...,ok
2,1000.0,62.5,c9 min 9 max 10,1739.124,1720.680,0.539,999.999,16,NaN,NaN,1.553,58.516,37.129,45.227,3126 Hz: -87.1 dBc; 358200 Hz: -65.2 dBc; 5556...,ok
3,1000.0,62.5,c8 min 9 max 10,3998.857,4021.177,0.567,999.999,16,NaN,NaN,-0.554,2001.998,162.764,1995.370,9738 Hz: -47.1 dBc; 9840 Hz: -63.8 dBc; 63060 ...,ok


In [3]:
import sys

sys.path.insert(0, str(PLL_DIR.parents[1]))  # repo root
from sw.lib.pll_util import bin_phase_noise

INK, GRID = "#52514e", "#e4e3df"
PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
OFFSET_AXIS = dict(type="log", tickvals=[1e2, 1e3, 1e4, 1e5, 1e6, 1e7, 1e8],
                   ticktext=["100 Hz", "1 kHz", "10 kHz", "100 kHz", "1 MHz", "10 MHz", "100 MHz"],
                   minor=dict(showgrid=True, gridcolor="#f1f0ed"))
N_BINS = 300  # points per plotted phase-noise curve (spread over F_start-F_stop); None = every measured point

# One colour per setting, in run order (not cycled: more than 8 settings -> split the run).
KEYS = list(zip(runs["f_vco"], runs["kvco_target"]))
COLORS = {k: PALETTE[i] for i, k in enumerate(KEYS)}
NAMES = {(r.f_vco, r.kvco_target): f"{r.f_vco / 1e6:g} MHz, |Kvco| {r.kvco_pred / 1e6:.0f} MHz/V "
                                   f"(c{r.vco_tune_coarse:.0f} min {r.vco_current_min:.0f} max {r.vco_current_max:.0f}), carrier {r.carrier_dbm:.1f} dBm"
         for r in runs.itertuples()}
N_DIV = {(r.f_vco, r.kvco_target): r.total_div for r in runs.itertuples()}
REF_TXT = (f"reference {runs['f_ref'].iloc[0] / 1e6:.4g} MHz" if runs["f_ref"].nunique() == 1  # input of the PLL
           else "reference per setting (see the table)")


def trace_of(key, kind):
    return traces[(traces["f_vco"] == key[0]) & (traces["kvco_target"] == key[1]) & (traces["kind"] == kind)]


def offset_axis(f_start, f_stop, f_axis):
    """x-axis settings for an offset plot shown from f_start to f_stop [Hz]; f_axis "log" or "lin"."""
    if f_axis == "log":
        return dict(OFFSET_AXIS, range=[np.log10(f_start), np.log10(f_stop)])
    return dict(type="linear", range=[f_start, f_stop], ticksuffix="Hz", exponentformat="SI")


def in_range(df, f_start, f_stop):
    """Rows of df with the offset (column freq) between f_start and f_stop, so the y axis fits the shown band."""
    return df[(df["freq"] >= f_start) & (df["freq"] <= f_stop)]


def binned(df, f_start, f_stop, f_axis):
    """The curve in df (columns freq, level_dbm) between f_start and f_stop, averaged into N_BINS bins spaced for
    f_axis, in linear power (pll_util.bin_phase_noise): display only, the jitter uses the measured points."""
    df = in_range(df, f_start, f_stop).sort_values("freq")
    if N_BINS is None or len(df) < 2:
        return df
    if f_axis == "log":
        edges = np.geomspace(f_start, f_stop, N_BINS + 1)
    else:
        edges = np.linspace(f_start, f_stop, N_BINS + 1)
    freq, level = bin_phase_noise(df["freq"], df["level_dbm"], edges)
    return pd.DataFrame({"freq": freq, "level_dbm": level})


def save(fig, name):
    """Interactive HTML and a static PNG (kaleido) next to the CSV, then show the figure."""
    base = RESULT_CSV.with_name(f"{RESULT_CSV.stem}_{name}")
    fig.write_html(base.with_suffix(".html"))
    try:
        fig.write_image(base.with_suffix(".png"), width=1100, height=fig.layout.height or 600, scale=2)
        print(f"saved {base}.html and .png")
    except Exception as e:  # kaleido >= 1.0 needs a Chrome/Chromium install (`plotly_get_chrome`)
        print(f"saved {base}.html; PNG export failed: {e}")
    fig.show()

In [4]:
F_start, F_stop = None, None  # offset from the carrier shown [Hz] (may be negative); None = edge of the trace
f_axis = "lin"  # "lin" or "log" (log: upper sideband only)

# Averaged wide spectrum around the carrier per setting: level relative to the carrier (trace maximum), in the trace's
# RBW (one sweep, not binned). Left: referred to the VCO (+ 20 log10 N); the carrier itself (within 10 RBW) is left
# out there, +20 log10 N only applies to the noise sidebands. Right: as measured on the pad.
from plotly.subplots import make_subplots

fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.08,
                    subplot_titles=("referred to the VCO (pad + 20 log10 N, carrier left out)", "measured on the pad"))
for key in KEYS:
    d = trace_of(key, "spectrum")
    if d.empty:
        continue
    d = d.assign(freq=d["freq"] - d["center"], level_dbc=d["level_dbm"] - d["level_dbm"].max())  # offset, rel. to carrier
    lo = d["freq"].min() if F_start is None else F_start
    hi = d["freq"].max() if F_stop is None else F_stop
    if f_axis == "log":
        lo = max(lo, d.loc[d["freq"] > 0, "freq"].min())
    d = in_range(d, lo, hi)
    sidebands = d[d["freq"].abs() >= 10 * d["rbw"]]
    sidebands = pd.concat([sidebands[sidebands["freq"] < 0], pd.DataFrame({"freq": [0.0], "level_dbc": [np.nan]}),
                           sidebands[sidebands["freq"] > 0]])  # gap at the carrier
    for col, curve, add_db, where in ((1, sidebands, 20 * np.log10(N_DIV[key]), "at the VCO"), (2, d, 0.0, "on the pad")):
        fig.add_scatter(x=curve["freq"], y=curve["level_dbc"] + add_db, mode="lines", name=NAMES[key],
                        legendgroup=NAMES[key], showlegend=col == 1, line=dict(color=COLORS[key], width=1.5),
                        row=1, col=col, hovertemplate=f"%{{x:.4s}}Hz: %{{y:.1f}} dBc {where}<extra></extra>")
rbw = traces.loc[traces["kind"] == "spectrum", "rbw"]
fig.update_layout(title=f"Spectrum around the carrier (RBW {rbw.iloc[0]:g} Hz), {REF_TXT}" if len(rbw)
                  else f"Spectrum around the carrier, {REF_TXT}",
                  template="plotly_white", height=600, legend=dict(orientation="h", y=-0.2),
                  hoverlabel=dict(namelength=-1))
if f_axis == "log":
    fig.update_xaxes(**OFFSET_AXIS)
else:
    fig.update_xaxes(type="linear", ticksuffix="Hz", exponentformat="SI")
fig.update_xaxes(title="offset from the carrier", gridcolor=GRID)
fig.update_yaxes(title="level relative to the carrier [dBc in RBW]", gridcolor=GRID)
save(fig, "spectrum")

saved /users/micas/wvandest/projects/lagd-meas/results/pll/pll_spectrum_kvco_S5_20261020_113911_spectrum.html and .png


In [5]:
F_start, F_stop = 1e3, 10e6  # offset range shown [Hz]
f_axis = "log"  # "log" or "lin"

# VCO-referred phase noise: the full curve (thick), its noise-only version with the spurs replaced by the local floor
# (dashed), the detected spurs (x at their measured peak, hover: spur power ON THE PAD in dBc: strong spurs are coupled
# in after the divider, so +20 log10 N does not apply to them), each span normalized to 1 Hz (thin, upper sideband,
# no ENBW correction: shows whether the spans line up).
# Curves binned to N_BINS points (linear power average, display only).
fig = go.Figure()
for key in KEYS:
    n_db = 20 * np.log10(N_DIV[key])
    for span, d in trace_of(key, "phase_noise").groupby("span"):
        d = d.assign(freq=d["freq"] - d["center"],  # offset from the carrier, normalized to 1 Hz
                     level_dbm=d["level_dbm"] - d["level_dbm"].max() - 10 * np.log10(d["rbw"]))
        d = binned(d[d["freq"] > 10 * d["rbw"]], F_start, F_stop, f_axis)
        fig.add_scatter(x=d["freq"], y=d["level_dbm"] + n_db,
                        mode="lines", line=dict(color=COLORS[key], width=0.8), opacity=0.35, showlegend=False,
                        legendgroup=NAMES[key],
                        hovertemplate=f"span {span:g} Hz<br>%{{x:.3s}}Hz: %{{y:.1f}} dBc/Hz<extra></extra>")
    lt_raw = in_range(trace_of(key, "L_trace"), F_start, F_stop)  # spur peaks: from the measured points
    lt = binned(trace_of(key, "L_trace"), F_start, F_stop, f_axis)
    ln = binned(trace_of(key, "L_noise"), F_start, F_stop, f_axis)
    if not lt.empty:
        fig.add_scatter(x=lt["freq"], y=lt["level_dbm"] + n_db, mode="lines", line=dict(color=COLORS[key], width=2.5),
                        name=NAMES[key], legendgroup=NAMES[key],
                        hovertemplate="full curve: %{x:.3s}Hz: %{y:.1f} dBc/Hz<extra></extra>")
    if not ln.empty:
        fig.add_scatter(x=ln["freq"], y=ln["level_dbm"] + n_db, mode="lines", showlegend=False, legendgroup=NAMES[key],
                        line=dict(color=INK, width=1.2, dash="dash"),
                        hovertemplate="noise only: %{x:.3s}Hz: %{y:.1f} dBc/Hz<extra></extra>")
    sp = in_range(trace_of(key, "spur"), F_start, F_stop)
    if not sp.empty and not lt_raw.empty:
        peak = np.interp(sp["freq"], lt_raw["freq"], lt_raw["level_dbm"]) + n_db
        fig.add_scatter(x=sp["freq"], y=peak, mode="markers", showlegend=False, legendgroup=NAMES[key],
                        marker=dict(symbol="x", size=11, color=COLORS[key], line=dict(width=1)),
                        customdata=sp["level_dbm"],
                        hovertemplate="spur at %{x:.4s}Hz: %{customdata:.1f} dBc on the pad<extra></extra>")
fig.update_layout(title=f"Phase noise referred to the VCO, pad + 20 log10 N (thick: full curve, dashed: noise only, x: spurs), {REF_TXT}",
                  template="plotly_white", height=550, legend=dict(orientation="h", y=-0.18),
                  hoverlabel=dict(namelength=-1))
fig.update_xaxes(title="offset from the carrier", gridcolor=GRID, **offset_axis(F_start, F_stop, f_axis))
fig.update_yaxes(title="L(f) referred to the VCO [dBc/Hz]", gridcolor=GRID)
save(fig, "phase_noise")

saved /users/micas/wvandest/projects/lagd-meas/results/pll/pll_spectrum_kvco_S5_20261020_113911_phase_noise.html and .png


In [6]:
F_start, F_stop = 1e3, 20e6  # offset range shown [Hz]

# Our full curve (thick) against the analyzer's Phase Noise application (K40, thin) on the same lock, both on the pad,
# with their spurs (x: ours, o: the K40 spur list; each drawn on its own curve); below the difference K40 - full curve per bin
# (same log-spaced bins for both). Only settings measured with phase_noise_mode=True. Jitter in the legend, both over
# the K40 band (pnm_band).
pnm_keys = [k for k in KEYS if len(trace_of(k, "L_pnmode"))]
if not pnm_keys:
    print("no phase-noise mode traces in this run (pll_spectrum_kvco.py with phase_noise_mode=True)")
else:
    edges = np.geomspace(F_start, F_stop, N_BINS + 1)
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.05, row_heights=[0.7, 0.3])
    for key in pnm_keys:
        r = runs[(runs["f_vco"] == key[0]) & (runs["kvco_target"] == key[1])].iloc[0]
        binned_l = {}
        for kind, name, width, jit in (("L_trace", "full curve", 2.5, r.time_jitter_trace_pnm_band_ps),
                                       ("L_pnmode", "K40", 1.2, r.time_jitter_pnm_ps)):
            d = in_range(trace_of(key, kind), F_start, F_stop).sort_values("freq")
            freq, level = bin_phase_noise(d["freq"], d["level_dbm"], edges)
            binned_l[kind] = pd.Series(level, index=np.digitize(freq, edges))
            fig.add_scatter(x=freq, y=level, mode="lines", line=dict(color=COLORS[key], width=width),
                            name=f"{NAMES[key]}: {name}, {jit:.2f} ps", legendgroup=NAMES[key],
                            hovertemplate=f"{name}: %{{x:.3s}}Hz: %{{y:.1f}} dBc/Hz<extra></extra>", row=1, col=1)
        for kind, curve, symbol, name in (("spur", "L_trace", "x", "spur (full curve)"),
                                          ("pnm_spur", "L_pnmode", "circle-open", "spur (K40)")):
            sp = in_range(trace_of(key, kind), F_start, F_stop)
            raw = in_range(trace_of(key, curve), F_start, F_stop).sort_values("freq")  # spur peak on its curve
            if sp.empty or raw.empty:
                continue
            fig.add_scatter(x=sp["freq"], y=np.interp(sp["freq"], raw["freq"], raw["level_dbm"]),
                            mode="markers", showlegend=False, legendgroup=NAMES[key],
                            marker=dict(symbol=symbol, size=10, color=COLORS[key], line=dict(width=1.5)),
                            customdata=sp["level_dbm"], row=1, col=1,
                            hovertemplate=f"{name} at %{{x:.4s}}Hz: %{{customdata:.1f}} dBc<extra></extra>")
        diff = (binned_l["L_pnmode"] - binned_l["L_trace"]).dropna()
        fig.add_scatter(x=np.sqrt(edges[diff.index - 1] * edges[diff.index]), y=diff.values, mode="lines",
                        line=dict(color=COLORS[key], width=1.5), showlegend=False, legendgroup=NAMES[key],
                        hovertemplate="%{x:.3s}Hz: %{y:+.2f} dB<extra></extra>", row=2, col=1)
    fig.add_hline(y=0, line=dict(color=GRID, width=1), row=2, col=1)
    fig.update_layout(title=f"Phase noise on the pad: full curve vs phase-noise mode (K40), {REF_TXT}",
                      template="plotly_white", height=750, legend=dict(orientation="h", y=-0.12),
                      hoverlabel=dict(namelength=-1))
    fig.update_xaxes(gridcolor=GRID, **offset_axis(F_start, F_stop, "log"))
    fig.update_xaxes(title="offset from the carrier", row=2)
    fig.update_yaxes(gridcolor=GRID)
    fig.update_yaxes(title="L(f) on the pad [dBc/Hz]", row=1)
    fig.update_yaxes(title="K40 - full curve [dB]", row=2)
    save(fig, "phase_noise_pnm")

saved /users/micas/wvandest/projects/lagd-meas/results/pll/pll_spectrum_kvco_S5_20261020_113911_phase_noise_pnm.html and .png


In [7]:
# RMS jitter (integration band of the run) per setting: the full curve (total, noise only, spur part).
# x axis: f_vco when the run sweeps the frequency, else the |Kvco| of the chosen setting.
ok = runs[runs["status"] == "ok"]
SWEEP_F = ok["f_vco"].nunique() > 1
X_COL, X_SCALE, X_TITLE = (("f_vco", 1e6, "f_vco [MHz]") if SWEEP_F else
                           ("kvco_pred", 1e6, "|Kvco| of the chosen setting [MHz/V]"))
ok = ok.sort_values(X_COL)
HOVER_DATA = np.c_[ok["vco_tune_coarse"], ok["vco_current_min"], ok["vco_current_max"], ok["kvco_pred"] / 1e6,
                   ok["carrier_dbm"]]
HOVER_SETTING = ("c%{customdata[0]:.0f} min %{customdata[1]:.0f} max %{customdata[2]:.0f}, "
                 "|Kvco| %{customdata[3]:.0f} MHz/V, carrier %{customdata[4]:.1f} dBm")
fig = go.Figure()
for col, name, color, dash in (("time_jitter_trace_ps", "full curve, total", PALETTE[0], "solid"),
                               ("time_jitter_noise_ps", "full curve, noise only", PALETTE[2], "solid"),
                               ("time_jitter_spur_ps", "full curve, spurs", PALETTE[3], "dot")):
    fig.add_scatter(x=ok[X_COL] / X_SCALE, y=ok[col], mode="lines+markers", name=name,
                    line=dict(color=color, width=2, dash=dash), marker=dict(size=9), customdata=HOVER_DATA,
                    hovertemplate=f"{name}: %{{y:.1f}} ps at %{{x:.0f}}<br>{HOVER_SETTING}<extra></extra>")
fig.update_layout(title=f"RMS time jitter (same on the pad and at the VCO), integrated {band[0]:.3g} Hz - {band[1] / 1e6:.3g} MHz, {REF_TXT}",
                  template="plotly_white", height=480, legend=dict(orientation="h", y=-0.2))
fig.update_xaxes(title=X_TITLE, type="log", gridcolor=GRID)
fig.update_yaxes(title="RMS time jitter [ps]", type="log", gridcolor=GRID)
save(fig, "jitter_vs_f" if SWEEP_F else "jitter_vs_kvco")

saved /users/micas/wvandest/projects/lagd-meas/results/pll/pll_spectrum_kvco_S5_20261020_113911_jitter_vs_kvco.html and .png


In [8]:
# PLL supply power and carrier level on the pad per setting (same x axis as the jitter plot).
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                    subplot_titles=["PLL supply power", "Carrier level on the pad (= PLL output)"])
fig.add_scatter(x=ok[X_COL] / X_SCALE, y=ok["pll_p"] * 1e3, mode="lines+markers", showlegend=False, row=1, col=1,
                line=dict(color=PALETTE[0], width=2), marker=dict(size=9), customdata=HOVER_DATA,
                hovertemplate=f"%{{y:.3f}} mW at %{{x:.0f}}<br>{HOVER_SETTING}<extra></extra>")
fig.add_scatter(x=ok[X_COL] / X_SCALE, y=ok["carrier_dbm"], mode="lines+markers", showlegend=False, row=2, col=1,
                line=dict(color=PALETTE[1], width=2), marker=dict(size=9), customdata=HOVER_DATA,
                hovertemplate=f"%{{y:.1f}} dBm at %{{x:.0f}}<br>{HOVER_SETTING}<extra></extra>")
fig.update_layout(title=f"PLL supply power and carrier level on the pad, {REF_TXT}", template="plotly_white", height=600)
fig.update_xaxes(type="log", gridcolor=GRID)
fig.update_xaxes(title=X_TITLE, row=2)
fig.update_yaxes(gridcolor=GRID)
fig.update_yaxes(title="power [mW]", rangemode="tozero", row=1)
fig.update_yaxes(title="carrier [dBm]", row=2)
save(fig, "power_carrier_vs_f" if SWEEP_F else "power_carrier_vs_kvco")

saved /users/micas/wvandest/projects/lagd-meas/results/pll/pll_spectrum_kvco_S5_20261020_113911_power_carrier_vs_kvco.html and .png
